# 6.3 LangChain/LangGraph Agent Basics

LangChain is a popular framework for building language model applications. **LangGraph** is a key component for building stateful graph-based multi-agent systems. It provides task control, observability, and graph-based orchestration.

In LangChain, an **Agent** is an entity that can use tools and execute actions based on user input. Key concepts:

- **LLM (Large Language Model)**: The reasoning engine that makes decisions.
- **Tool**: A function or API that the Agent can call to execute tasks.
- **@tool decorator**: A LangChain utility that converts a Python function into a tool callable by the LLM.
- **Orchestrator**: In LangGraph, the `StateGraph` manages state transitions and control flow between multiple nodes (Agents or functions).

---

## `@tool` Decorator Explained

The `@tool` decorator converts a Python function into a "tool callable by the Agent". It exposes the function name, description, and parameter information so the LLM can use it.

### Example:
```python
from langchain_core.tools import tool

@tool
def my_function(input: str) -> str:
    """This is the tool description, which will be read by the LLM"""
    return f"Processing result: {input}"
```

## Example 1: Hello World - Basic Tool Usage

This example defines a simple tool and lets the Agent call it.

In [ ]:
#!pip install langchain_core langchain_openai langchain langgraph numpy

In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent  # new import
from langchain_core.messages import HumanMessage  # for input messages

# Define a tool
@tool
def greet_user(name: str) -> str:
    """Greet the user by name"""
    return f"Hello, {name}! Nice to meet you."

# Using io.solutions model (OpenAI-compatible interface)
llm = ChatOpenAI(
    base_url="https://api.intelligence.io.solutions/api/v1",
    api_key="80e68c38-22cb-4f71-9377-0768c4d7fe15",
    model="moonshotai/Kimi-K2.6",  # Replace with your model ID
    temperature=0.1,
    top_p=0.9,
    max_tokens=1024
)

# Create the agent
tools = [greet_user]
agent = create_agent(
    model=llm,  # LLM instance
    tools=tools,
    system_prompt="You are a helpful assistant. Use tools to complete user requests."  # system prompt string
)

# Execute (invoke, no Executor needed)
result = agent.invoke({
    "messages": [HumanMessage(content="Say hello to Alice")]
})

In [ ]:
result["messages"][-1].content

> This demonstrates the basic Agent usage pattern: LLM parses user request -> calls `greet_user` tool -> returns result.

## Example 2: Multi-Tool Agent - Weather Query + Clothing Suggestion

This example uses two tools: one to get weather information, and another to suggest clothing based on the weather. The Agent performs multi-step reasoning and tool calling.

In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent  # new import: langchain.agents
from langchain_core.messages import HumanMessage  # for input messages

# Tool 1: Simulate getting weather
@tool
def get_weather(city: str) -> str:
    """Get weather information for a city (simulated)"""
    weather_data = {"Beijing": "Sunny, 25C", "Shanghai": "Rainy, 20C", "Guangzhou": "Cloudy, 30C"}
    return weather_data.get(city, "Weather data not available")

# Tool 2: Suggest clothing based on weather
@tool
def suggest_clothing(weather_info: str) -> str:
    """Suggest clothing based on weather information"""
    if "Rainy" in weather_info:
        return "Bring an umbrella and wear a waterproof jacket."
    elif "25" in weather_info or "30" in weather_info:
        return "It's warm - wear light clothing like shorts and a t-shirt."
    else:
        return "Moderate weather - a long-sleeve shirt should be fine."

# Using io.solutions model (OpenAI-compatible interface)
llm = ChatOpenAI(
    base_url="https://api.intelligence.io.solutions/api/v1",
    api_key="80e68c38-22cb-4f71-9377-0768c4d7fe15",
    model="moonshotai/Kimi-K2.6",  # Replace with your model ID
    temperature=0.1,
    top_p=0.9,
    max_tokens=1024
)

tools = [get_weather, suggest_clothing]

# Create agent (with tool usage)
system_prompt = "You are a helpful assistant. Use tools to complete user requests, such as querying weather and suggesting clothing."
agent_executor = create_agent(
    llm,
    tools,
    system_prompt=system_prompt,  # system prompt (string)
    debug=True  # print debug info to see tool calling
)

# Execute task
query = "What's the weather in Beijing? What should I wear today?"
result = agent_executor.invoke({
    "messages": [HumanMessage(content=query)]  # input message
})

# Output response (last message content)
print(result["messages"][-1].content)
# Example output: Beijing is sunny, 25C. Wear light clothing like shorts and a t-shirt.

> This Agent completes the following process:
> 1. Calls `get_weather("Beijing")` to get weather info;
> 2. Passes the result to `suggest_clothing`;
> 3. Outputs the final recommendation.

## Summary

| Concept | Explanation |
|------|------|
| `@tool` | Converts a function into a tool that the LLM can discover and call |
| Agent capabilities | LLM decides which tools to call based on input and tool descriptions |
| LangGraph usage | For complex scenarios, use graphs to manage multiple Agents and state (loops, conditional branches) |

Through LangChain's `@tool` and Agent, you can build "perceive-think-act" capabilities, enabling autonomous task execution.

# AirSim API as Agent Tools

This is the key bridge connecting LLM Agents to the AirSim simulator. We need to wrap AirSim API calls as "tool" functions that the LLM can discover and call.

The following are the tool implementations in Python. Using LangChain/LangGraph's `@tool` decorator, the framework automatically extracts the **function name, parameters, types**, and **docstring** to generate a tool schema for the LLM. These tools serve as the interface between the agent and the simulation environment.

## Concepts

- **Tool wrapping**: Exposing APIs as high-level functions for the language model.
- **@tool decorator benefits**:
  - Generates tool descriptions (Tool Schema) automatically.
  - Enables the LLM to call tools based on the task.
  - Implements the "action" step in the "perceive-think-act" loop.
- **Purpose**: The Agent calls these tools to interact with the AirSim environment, enabling both perception (getting images/state) and action (flight control).

> Note: Good docstrings are essential - the LLM relies on them to understand tool usage and parameters.

In [ ]:
import sys
sys.path.append('..')

import airsim
from langchain_core.tools import tool
from typing import Dict

# Initialize AirSim client
client = airsim.MultirotorClient()
client.confirmConnection()

@tool
def takeoff(drone_id: str) -> str:
    """
    Command the drone to take off.
    :param drone_id: The drone to control (e.g., 'Drone1').
    """
    print(f"Executing: takeoff for {drone_id}")
    # Must enable API control first
    client.enableApiControl(True, vehicle_name=drone_id)
    client.armDisarm(True, vehicle_name=drone_id)
    # Async takeoff, wait for completion
    client.takeoffAsync(vehicle_name=drone_id).join()
    return f"Drone {drone_id} has taken off successfully."

@tool
def fly_to_position(drone_id: str, x: float, y: float, z: float) -> str:
    """
    Command the drone to fly to a position at 5 m/s velocity in NED coordinates.
    Z is negative for above ground (higher altitude = more negative Z).
    :param drone_id: The drone to control.
    :param x: Target position X coordinate.
    :param y: Target position Y coordinate.
    :param z: Target position Z coordinate (negative = higher altitude).
    """
    print(f"Executing: fly_to_position for {drone_id} to ({x}, {y}, {z})")
    client.moveToPositionAsync(x, y, z, 5, vehicle_name=drone_id).join()
    return f"Drone {drone_id} has arrived at target position ({x}, {y}, {z})."

@tool
def get_drone_state(drone_id: str) -> Dict:
    """
    Get the drone's current state, including position and orientation.
    :param drone_id: The drone to query.
    """
    print(f"Executing: get_drone_state for {drone_id}")
    state = client.getMultirotorState(vehicle_name=drone_id)
    # Return dictionary for LLM processing
    return {
        "position": {
            "x_val": state.kinematics_estimated.position.x_val,
            "y_val": state.kinematics_estimated.position.y_val,
            "z_val": state.kinematics_estimated.position.z_val,
        },
        "orientation": {
            "w_val": state.kinematics_estimated.orientation.w_val,
            "x_val": state.kinematics_estimated.orientation.x_val,
            "y_val": state.kinematics_estimated.orientation.y_val,
            "z_val": state.kinematics_estimated.orientation.z_val,
        }
    }

With these tools defined, the LLM has a clean, high-level interface to the AirSim API.

The LLM doesn't need to know about `airsim.MultirotorClient` directly - it simply calls `fly_to_position` based on the tool's docstring description.